# 🚀 Projeto Prático — Sistema de Cadastro e Login

## Projeto Thiago Cardoso Santos — IA com Impacto
### Curso Júnior | Turma 1 | Lions Clube de Osório

Neste exercício vamos desenvolver uma aplicação web utilizando Python.

A aplicação permitirá:

- Criar uma conta
- Armazenar o usuário em um banco de dados
- Realizar login
- Validar usuário e senha
- Acessar uma área protegida
- Realizar logout

Ao final do exercício teremos uma aplicação funcionando em nossa própria máquina.

> **Pergunta da aula:** uma tela mostrando "Login realizado" prova que o usuário realmente foi autenticado?

Vamos descobrir construindo e testando o sistema de ponta a ponta.

In [1]:
# ============================================================
# Célula 1 — Projeto Prático: Sistema de Cadastro e Login
# ============================================================
#
# Projeto Thiago Cardoso Santos — IA com Impacto
# Curso Júnior — Turma 1
#
# Neste exercício vamos criar:
#
# Python → SQLite → Streamlit → Cadastro → Login
#        → Área Protegida → Logout
#
# Primeiro vamos verificar se o Python está funcionando.

import sys

print("Versão do Python:")
print(sys.version)

print("\n✅ Python funcionando!")

Versão do Python:
3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]

✅ Python funcionando!


In [2]:
# ============================================================
# Célula 2 — Verificar o SQLite
# ============================================================

# Importa o SQLite.
# Ele já acompanha o Python, portanto normalmente
# não precisamos instalar nada.

import sqlite3

# Mostra a versão instalada.
print("Versão do SQLite:")
print(sqlite3.sqlite_version)

print("\n✅ SQLite funcionando!")

Versão do SQLite:
3.45.3

✅ SQLite funcionando!


In [3]:
# ============================================================
# Célula 3 — Instalar o Streamlit
# ============================================================

# O Streamlit transforma nosso código Python
# em uma aplicação que podemos abrir no navegador.

%pip install streamlit -q

print("✅ Streamlit instalado!")

Note: you may need to restart the kernel to use updated packages.
✅ Streamlit instalado!



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# ============================================================
# Célula 4 — Criar o banco
# ============================================================

# Importa o SQLite
import sqlite3

# Conecta ao banco.
# Se usuarios.db não existir, o Python cria automaticamente.
conexao = sqlite3.connect("usuarios.db")

# Cria um cursor para executar comandos SQL
cursor = conexao.cursor()

# Cria a tabela de usuários caso ela ainda não exista
cursor.execute("""
CREATE TABLE IF NOT EXISTS usuarios (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    senha_hash TEXT NOT NULL,
    criado_em DATETIME DEFAULT CURRENT_TIMESTAMP
)
""")

# Salva as alterações
conexao.commit()

# Fecha a conexão
conexao.close()

print("✅ Banco de dados criado!")

✅ Banco de dados criado!


In [5]:
# ============================================================
# Célula 5 — Verificar se a tabela foi criada
# ============================================================

import sqlite3

# Abre novamente nosso banco
conexao = sqlite3.connect("usuarios.db")

cursor = conexao.cursor()

# Consulta as tabelas existentes
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
""")

# Guarda o resultado
tabelas = cursor.fetchall()

# Fecha a conexão
conexao.close()

# Mostra as tabelas encontradas
print("Tabelas encontradas:")

for tabela in tabelas:
    print("→", tabela[0])

print("\n✅ Banco verificado!")

Tabelas encontradas:
→ usuarios
→ sqlite_sequence

✅ Banco verificado!


In [ ]:
%%writefile app.py
# ============================================================
# app.py — Aplicação Streamlit (login + jogos em páginas)
# ============================================================

import streamlit as st
import sqlite3
import hashlib


st.set_page_config(
    page_title="IA com Impacto",
    page_icon="🚀",
    layout="centered"
)


# ------------------------------------------------------------
# FUNÇÃO PARA PROTEGER A SENHA
# ------------------------------------------------------------

def proteger_senha(senha):
    return hashlib.sha256(
        senha.encode()
    ).hexdigest()


# ------------------------------------------------------------
# FUNÇÃO PARA CADASTRAR UM USUÁRIO
# ------------------------------------------------------------

def cadastrar_usuario(nome, email, senha):

    conexao = sqlite3.connect("usuarios.db")
    cursor = conexao.cursor()
    senha_hash = proteger_senha(senha)

    try:
        cursor.execute("""
        INSERT INTO usuarios (
            nome,
            email,
            senha_hash
        )
        VALUES (?, ?, ?)
        """, (
            nome,
            email,
            senha_hash
        ))
        conexao.commit()
        return True

    except sqlite3.IntegrityError:
        return False

    finally:
        conexao.close()


# ------------------------------------------------------------
# FUNÇÃO PARA REALIZAR LOGIN
# ------------------------------------------------------------

def fazer_login(email, senha):

    conexao = sqlite3.connect("usuarios.db")
    cursor = conexao.cursor()
    senha_hash = proteger_senha(senha)

    cursor.execute("""
    SELECT
        id,
        nome,
        email
    FROM usuarios
    WHERE email = ?
    AND senha_hash = ?
    """, (
        email,
        senha_hash
    ))

    usuario = cursor.fetchone()
    conexao.close()
    return usuario


# ------------------------------------------------------------
# CRIAR A SESSÃO
# ------------------------------------------------------------

if "usuario" not in st.session_state:
    st.session_state.usuario = None


# ------------------------------------------------------------
# USUÁRIO NÃO ESTÁ LOGADO  → tela de Login / Cadastro
# ------------------------------------------------------------

if st.session_state.usuario is None:

    st.title("🚀 IA com Impacto")
    st.write("### Projeto Thiago Cardoso Santos")
    st.caption("Lions Clube de Osório — Turma 1")
    st.divider()

    aba_login, aba_cadastro = st.tabs([
        "🔐 Login",
        "👤 Cadastro"
    ])

    # ---------------- TELA DE LOGIN ----------------
    with aba_login:

        st.subheader("💙🖤🤍Gremio💙🖤🤍")

        email = st.text_input("E-mail", key="login_email")
        senha = st.text_input("Senha", type="password", key="login_senha")

        if st.button("Entrar", key="botao_login"):

            usuario = fazer_login(email, senha)

            if usuario:
                st.session_state.usuario = usuario
                st.rerun()
            else:
                st.error("❌ E-mail ou senha incorretos.")

    # ---------------- TELA DE CADASTRO ----------------
    with aba_cadastro:

        st.subheader("Criar Conta")

        nome = st.text_input("Nome completo", key="cadastro_nome")
        novo_email = st.text_input("E-mail", key="cadastro_email")
        nova_senha = st.text_input("Senha", type="password", key="cadastro_senha")
        confirmar_senha = st.text_input("Confirmar senha", type="password", key="confirmar_senha")

        if st.button("Criar Conta", key="botao_cadastro"):

            if (
                not nome
                or not novo_email
                or not nova_senha
                or not confirmar_senha
            ):
                st.warning("⚠️ Preencha todos os campos.")

            elif nova_senha != confirmar_senha:
                st.warning("⚠️ As senhas não são iguais.")

            else:
                resultado = cadastrar_usuario(nome, novo_email, nova_senha)

                if resultado:
                    st.success("✅ Conta criada com sucesso!")
                    st.info("Agora utilize a aba Login para entrar.")
                else:
                    st.error("❌ Este e-mail já está cadastrado.")


# ------------------------------------------------------------
# USUÁRIO LOGADO  → ÁREA PROTEGIDA com UMA PÁGINA POR JOGO
# ------------------------------------------------------------

else:

    usuario = st.session_state.usuario

    # Barra lateral: quem está logado + botão Sair (vale em todas as páginas)
    st.sidebar.title("🚀 IA com Impacto")
    st.sidebar.write(f"👤 **{usuario[1]}**")
    st.sidebar.caption(f"📧 {usuario[2]}")

    if st.sidebar.button("🚪 Sair"):
        st.session_state.usuario = None
        st.rerun()

    st.sidebar.divider()

    # Importa os jogos
    import jogos

    # Cada jogo = uma PÁGINA (aparece no menu da esquerda)
    paginas = [
        st.Page(jogos.pagina_inicio,            title="Inicio",                 icon="🏠", default=True),
        st.Page(jogos.jogo_adivinhe,            title="Adivinhe o Numero",      icon="🔢"),
        st.Page(jogos.jogo_pedra_papel_tesoura, title="Pedra, Papel e Tesoura", icon="✊"),
        st.Page(jogos.jogo_dados,               title="Duelo de Dados",         icon="🎲"),
        st.Page(jogos.jogo_quiz,                title="Quiz Relampago",         icon="⚡"),
        st.Page(jogos.jogo_forca,               title="Forca",                  icon="🪢"),
    ]

    pagina = st.navigation(paginas)
    pagina.run()


In [ ]:
%%writefile jogos.py
# ============================================================
# jogos.py — Os 5 jogos da turma, cada um em sua PÁGINA
# ============================================================
#
# São os mesmos jogos de antes; o que muda é a navegação:
# cada jogo agora é uma PÁGINA (st.Page) no menu da esquerda,
# montado pelo app.py com st.navigation.
#
# A "memória" de cada jogo (número secreto, placar, letras...)
# continua no st.session_state — por isso o progresso se
# mantém quando você troca de página.
# ============================================================

import random

import streamlit as st


# ------------------------------------------------------------
# PÁGINA INICIAL (abre quando o aluno entra)
# ------------------------------------------------------------

def pagina_inicio():

    st.title("🚀 IA com Impacto")
    st.caption("Projeto Thiago Cardoso Santos — Lions Clube de Osorio — Turma 1")

    st.divider()

    usuario = st.session_state.get("usuario")
    if usuario:
        st.success(f"✅ Bem-vindo, {usuario[1]}!")

    st.subheader("🎮 Jogos da Turma")

    st.write(
        "Escolha um jogo no **menu da esquerda**. "
        "Cada jogo tem a sua **propria pagina**."
    )

    st.write("🔢 Adivinhe o Numero")
    st.write("✊ Pedra, Papel e Tesoura")
    st.write("🎲 Duelo de Dados")
    st.write("⚡ Quiz Relampago")
    st.write("🪢 Forca")


# ------------------------------------------------------------
# JOGO 1 — ADIVINHE O NÚMERO
# ------------------------------------------------------------

def jogo_adivinhe():

    st.subheader("🔢 Adivinhe o Número")
    st.write("Sorteei um número de 1 a 20. Você tem 5 tentativas.")

    if "adv_segredo" not in st.session_state:
        st.session_state.adv_segredo = random.randint(1, 20)
        st.session_state.adv_tentativas = 0
        st.session_state.adv_dicas = []
        st.session_state.adv_fim = False

    if not st.session_state.adv_fim:

        with st.form("form_adivinhe", clear_on_submit=True):
            palpite = st.number_input(
                "Seu número", min_value=1, max_value=20, step=1
            )
            chutar = st.form_submit_button("Chutar")

        if chutar:
            st.session_state.adv_tentativas += 1
            segredo = st.session_state.adv_segredo
            n = st.session_state.adv_tentativas

            if palpite == segredo:
                st.session_state.adv_dicas.append(f"{n}/5: {palpite} — ACERTOU! 🎉")
                st.session_state.adv_fim = True
            elif palpite < segredo:
                st.session_state.adv_dicas.append(f"{n}/5: {palpite} — o número é MAIOR.")
            else:
                st.session_state.adv_dicas.append(f"{n}/5: {palpite} — o número é MENOR.")

            if not st.session_state.adv_fim and n >= 5:
                st.session_state.adv_dicas.append(f"Acabaram as tentativas. O número era {segredo}.")
                st.session_state.adv_fim = True

    for dica in st.session_state.adv_dicas:
        st.write(dica)

    if st.button("🔄 Novo jogo", key="adv_novo"):
        del st.session_state.adv_segredo
        st.rerun()


# ------------------------------------------------------------
# JOGO 2 — PEDRA, PAPEL E TESOURA
# ------------------------------------------------------------

def jogo_pedra_papel_tesoura():

    st.subheader("✊ Pedra, Papel e Tesoura")
    st.write("Melhor de 3: quem ganhar 2 rounds vence.")

    if "ppt_vitorias" not in st.session_state:
        st.session_state.ppt_vitorias = 0
        st.session_state.ppt_derrotas = 0
        st.session_state.ppt_ultimo = ""

    vitorias = st.session_state.ppt_vitorias
    derrotas = st.session_state.ppt_derrotas
    terminou = vitorias == 2 or derrotas == 2

    opcoes = ["pedra", "papel", "tesoura"]
    ganha_de = [("pedra", "tesoura"), ("papel", "pedra"), ("tesoura", "papel")]

    if not terminou:

        colunas = st.columns(3)
        for coluna, opcao in zip(colunas, opcoes):
            if coluna.button(opcao.capitalize(), key=f"ppt_{opcao}"):
                pc = random.choice(opcoes)
                if opcao == pc:
                    resultado = "Empate!"
                elif (opcao, pc) in ganha_de:
                    st.session_state.ppt_vitorias += 1
                    resultado = "Você ganhou o round!"
                else:
                    st.session_state.ppt_derrotas += 1
                    resultado = "O computador ganhou o round!"
                st.session_state.ppt_ultimo = f"Você: {opcao} | Computador: {pc} — {resultado}"
                st.rerun()

    if st.session_state.ppt_ultimo:
        st.write(st.session_state.ppt_ultimo)

    st.write(
        f"Placar: você {st.session_state.ppt_vitorias} x "
        f"{st.session_state.ppt_derrotas} computador"
    )

    if st.session_state.ppt_vitorias == 2:
        st.success("VOCÊ VENCEU o jogo! 🏆")
    elif st.session_state.ppt_derrotas == 2:
        st.error("O computador venceu. Tente de novo!")

    if st.button("🔄 Novo jogo", key="ppt_novo"):
        del st.session_state.ppt_vitorias
        st.rerun()


# ------------------------------------------------------------
# JOGO 3 — DUELO DE DADOS
# ------------------------------------------------------------

def jogo_dados():

    st.subheader("🎲 Duelo de Dados")
    st.write("3 rodadas. Quem tirar o maior número ganha o ponto.")

    if "dados_rodada" not in st.session_state:
        st.session_state.dados_rodada = 0
        st.session_state.dados_voce = 0
        st.session_state.dados_pc = 0
        st.session_state.dados_historico = []

    if st.session_state.dados_rodada < 3:

        if st.button(f"Rolar o dado (rodada {st.session_state.dados_rodada + 1}/3)", key="dados_rolar"):
            d_voce = random.randint(1, 6)
            d_pc = random.randint(1, 6)
            st.session_state.dados_rodada += 1

            if d_voce > d_pc:
                st.session_state.dados_voce += 1
                quem = "Ponto seu!"
            elif d_pc > d_voce:
                st.session_state.dados_pc += 1
                quem = "Ponto do computador!"
            else:
                quem = "Empate nesta rodada!"

            st.session_state.dados_historico.append(
                f"Rodada {st.session_state.dados_rodada}: você {d_voce} x {d_pc} computador — {quem}"
            )
            st.rerun()

    for linha in st.session_state.dados_historico:
        st.write(linha)

    st.write(
        f"Placar: você {st.session_state.dados_voce} x "
        f"{st.session_state.dados_pc} computador"
    )

    if st.session_state.dados_rodada == 3:
        if st.session_state.dados_voce > st.session_state.dados_pc:
            st.success("VITÓRIA! 🏆")
        elif st.session_state.dados_pc > st.session_state.dados_voce:
            st.error("DERROTA!")
        else:
            st.info("EMPATE geral!")

    if st.button("🔄 Novo jogo", key="dados_novo"):
        del st.session_state.dados_rodada
        st.rerun()


# ------------------------------------------------------------
# JOGO 4 — QUIZ RELÂMPAGO
# ------------------------------------------------------------

def jogo_quiz():

    st.subheader("⚡ Quiz Relâmpago")
    st.write("Responda sem acento e tudo em minúsculo.")

    perguntas = [
        ("Quanto é 7 x 8?", "56"),
        ("Qual a capital do Brasil? (sem acento)", "brasilia"),
        ("Qual função mostra texto na tela em Python?", "print"),
        ("Quanto é 2 ** 3 (2 elevado a 3)?", "8"),
    ]

    with st.form("form_quiz"):
        respostas = [
            st.text_input(f"{i}) {pergunta}", key=f"quiz_{i}")
            for i, (pergunta, _) in enumerate(perguntas, 1)
        ]
        conferir = st.form_submit_button("Conferir respostas")

    if conferir:
        pontos = 0
        for (pergunta, certa), resposta in zip(perguntas, respostas):
            if resposta.strip().lower() == certa:
                pontos += 1
                st.write(f"✅ {pergunta} — certo!")
            else:
                st.write(f"❌ {pergunta} — a resposta era: {certa}")
        st.success(f"Você fez {pontos} de {len(perguntas)} pontos!")


# ------------------------------------------------------------
# JOGO 5 — FORCA
# ------------------------------------------------------------

def jogo_forca():

    st.subheader("🪢 Forca")
    st.write("Adivinhe a palavra, uma letra por vez. Você pode errar 6 vezes.")

    palavras = ["python", "jupyter", "codigo", "aegishub", "notebook"]
    max_erros = 6

    if "forca_palavra" not in st.session_state:
        st.session_state.forca_palavra = random.choice(palavras)
        st.session_state.forca_acertadas = set()
        st.session_state.forca_erradas = set()

    palavra = st.session_state.forca_palavra
    acertadas = st.session_state.forca_acertadas
    erradas = st.session_state.forca_erradas

    ganhou = all(c in acertadas for c in palavra)
    perdeu = len(erradas) >= max_erros

    if not ganhou and not perdeu:
        with st.form("form_forca", clear_on_submit=True):
            letra = st.text_input("Diga uma letra", max_chars=1)
            tentar = st.form_submit_button("Tentar")

        if tentar and letra.strip():
            letra = letra.strip().lower()[0]
            if letra in palavra:
                acertadas.add(letra)
            else:
                erradas.add(letra)
            st.rerun()

    mostra = " ".join(c if c in acertadas else "_" for c in palavra)
    st.markdown(f"### {mostra}")
    st.write(f"Erros: {len(erradas)}/{max_erros}  {' '.join(sorted(erradas))}")

    if ganhou:
        st.success(f"PARABÉNS! Você adivinhou: {palavra} 🎉")
    elif perdeu:
        st.error(f"Fim de jogo! A palavra era: {palavra}")

    if st.button("🔄 Novo jogo", key="forca_novo"):
        del st.session_state.forca_palavra
        st.rerun()


In [8]:
# ============================================================
# Célula 7 — Verificar os arquivos criados
# ============================================================

import os

# Mostra a pasta onde o Notebook está executando
print("Pasta atual:")
print(os.getcwd())

print("\nArquivos do projeto:")

# Mostra os arquivos existentes
for arquivo in os.listdir():
    print("→", arquivo)

Pasta atual:
C:\Users\Heleno Borba\PROJETOS_PYTHON\TURMA_1\curso-junior

Arquivos do projeto:
→ .ipynb_checkpoints
→ 00_ambiente.ipynb
→ 01_pensamento_logico.ipynb
→ 02_primeira_automacao.ipynb
→ 03_ia_na_pratica.ipynb
→ 04_primeiro_workflow.ipynb
→ app.py
→ jogos.py
→ paginawebte.ipynb
→ relatorios
→ Untitled.ipynb
→ usuarios.db


In [9]:
# ============================================================
# Célula 8 — Executar a aplicação em segundo plano
# ============================================================

import subprocess
import sys

# Inicia o Streamlit como outro processo.
# Assim o Jupyter continua livre para executar
# as próximas células.

processo_streamlit = subprocess.Popen([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    "app.py",
    "--server.port",
    "8501"
])

print("✅ Aplicação iniciada!")
print("🌐 Acesse: http://localhost:8501")
print("Processo:", processo_streamlit.pid)

✅ Aplicação iniciada!
🌐 Acesse: http://localhost:8501
Processo: 43132


In [10]:
# ============================================================
# Célula 9 — Verificar os usuários cadastrados
# ============================================================

import sqlite3

# Abre o mesmo banco utilizado pela aplicação
conexao = sqlite3.connect("usuarios.db")

cursor = conexao.cursor()

# Busca os usuários cadastrados
cursor.execute("""
SELECT
    id,
    nome,
    email,
    senha_hash,
    criado_em
FROM usuarios
""")

usuarios = cursor.fetchall()

# Fecha a conexão
conexao.close()

# Mostra quantos usuários foram encontrados
print("Total de usuários:", len(usuarios))

# Mostra cada usuário cadastrado
for usuario in usuarios:

    print("\n--------------------------------")

    print("ID:", usuario[0])
    print("Nome:", usuario[1])
    print("E-mail:", usuario[2])
    print("Senha protegida:", usuario[3])
    print("Criado em:", usuario[4])

print("\n--------------------------------")
print("✅ Consulta concluída!")

Total de usuários: 1

--------------------------------
ID: 1
Nome: Heleno
E-mail: helenoborba17@gmail.com
Senha protegida: d36c2e59ddde918939f57c171035c4cb43369064c1aa8017a780091f4db95502
Criado em: 2026-10-20 13:41:05

--------------------------------
✅ Consulta concluída!


In [ ]:
# ============================================================
# Célula 10 — Teste Final
# ============================================================
#
# Agora vamos validar nosso sistema de ponta a ponta.
#
# Execute manualmente os testes abaixo:
#
# TESTE 1
# Criar uma nova conta.
# Resultado esperado:
# ✅ Usuário cadastrado.
#
# TESTE 2
# Tentar cadastrar novamente o mesmo e-mail.
# Resultado esperado:
# ✅ Cadastro recusado.
#
# TESTE 3
# Fazer Login com a senha correta.
# Resultado esperado:
# ✅ Acesso autorizado.
#
# TESTE 4
# Fazer Login com uma senha errada.
# Resultado esperado:
# ✅ Acesso recusado.
#
# TESTE 5
# Entrar e clicar em "Sair".
# Resultado esperado:
# ✅ Sessão encerrada.
#
# REGRA DA AULA:
#
# Código escrito não significa código funcionando.
#
# Precisamos:
#
# EXECUTAR → TESTAR → MEDIR → COMPROVAR
#
# ============================================================

print("🧪 Testes E2E prontos para execução.")